In [1]:
import yaml

from jinja2 import Template
from langsmith import Client as LSClient

### Prompt Rendering

In [6]:
def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{question}
"""

    return prompt

In [7]:
preprocessed_context = "- a \n- b "
question = "What is a?"

In [8]:
print(build_prompt(preprocessed_context, question))


You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
- a 
- b 

Question:
What is a?



### Jinja Template

In [9]:
jinja_template = Template(
"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{{ preprocessed_context }}

Question:
{{ question }}
"""
)

In [10]:
rendered_template = jinja_template.render(preprocessed_context=preprocessed_context, question=question)

In [11]:
print(rendered_template)


You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
- a 
- b 

Question:
What is a?


In [12]:
def  build_prompt_jinja(preprocessed_context, question):
    prompt = """
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{{ preprocessed_context }}

Question:
{{ question }}
"""
    jinja_template_prompt = Template(prompt)

    return jinja_template_prompt.render(preprocessed_context=preprocessed_context, question=question)


In [13]:
print(build_prompt_jinja("- item", "question?"))


You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
- item

Question:
question?


In [17]:
def from_template_config(yaml_path: str, prompt: str):

    with open(yaml_path, 'r') as file:
        config = yaml.safe_load(file)

    template_content = config["prompts"][prompt]

    return Template(template_content)

In [18]:
def  build_prompt_jinja(prompts_file, prompt, preprocessed_context, question):
    
    template = from_template_config(prompts_file, prompt)

    return template.render(preprocessed_context=preprocessed_context, question=question)


In [20]:
print(build_prompt_jinja("prompts/retrieval_generation.yaml", "retrieval_generation", preprocessed_context, question))

You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
- a 
- b 

Question:
What is a?


### Prompt Registries

In [21]:
ls_client = LSClient()

In [22]:
ls_template = ls_client.pull_prompt("retrieval_generation")

/Users/vaidasarmonas/Learning/AI-Engineering-Bootcamp/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [23]:
ls_template

ChatPromptTemplate(input_variables=[], input_types={}, partial_variables={}, metadata={'lc_hub_owner': '-', 'lc_hub_repo': 'retrieval_generation', 'lc_hub_commit_hash': '0b4b809c513ade9cf1814e2bc1abe418402c3962493530a97427977ab3ce62f0'}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a shopping assistant that can answer questions about the products in stock.\nYou will be given a question and a list of context.\n\n Instructions:\n- Answer the question based on the provided context only.\n- Never use word context and refer to it as the available products.\n- Do not use markdown formatting.\n\nContext:\n{{ preprocessed_context }}\n\nQuestion:\n{{ question }}'), additional_kwargs={})])

In [24]:
print(ls_template.messages[0].prompt.template)

You are a shopping assistant that can answer questions about the products in stock.
You will be given a question and a list of context.

 Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{{ preprocessed_context }}

Question:
{{ question }}


In [29]:
def  build_prompt_registry(prompt, ls_client, preprocessed_context, question):
    
    template_text = ls_client.pull_prompt(prompt).messages[0].prompt.template
    
    template = Template(template_text)

    return template.render(preprocessed_context=preprocessed_context, question=question)

In [30]:
print(build_prompt_registry(prompt="retrieval_generation", ls_client=ls_client, preprocessed_context=preprocessed_context, question=question))

You are a shopping assistant that can answer questions about the products in stock.
You will be given a question and a list of context.

 Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
- a 
- b 

Question:
What is a?
